# Phase 2 — Data Cleaning: Ames Housing

This notebook investigates and handles missing values in the Ames Housing dataset. The original raw CSV is preserved.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../../data/raw/ames_housing/train.csv')
OUTPUT_DIR = Path('../../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print('Missing cells:', int(df.isna().sum().sum()))
print('Missing target values:', int(df['SalePrice'].isna().sum()))

## Missing-value analysis

Review missing counts and percentages before choosing cleaning rules.

In [ ]:
missing = df.isna().sum()
missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': (missing / len(df) * 100).round(2),
    'dtype': df.dtypes.astype(str)
})
missing_report = missing_report[missing_report['missing_count'] > 0]
missing_report = missing_report.sort_values('missing_count', ascending=False)
display(missing_report)

## Cleaning strategy

Some missing categorical values represent the absence of a feature, such as a garage, basement, pool, or fireplace. Other missing values represent unknown measurements. These cases are handled separately. Model-specific imputers should be fitted on training data only.

In [ ]:
cleaned = df.copy()

# Missing categories that indicate the feature is absent.
no_feature_columns = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence',
    'MasVnrType', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'BsmtQual', 'BsmtCond'
]

for col in no_feature_columns:
    if col in cleaned.columns:
        cleaned[col] = cleaned[col].fillna('None')

# Estimate missing lot frontage from the property's neighborhood,
# then fall back to the overall median.
if 'LotFrontage' in cleaned.columns and 'Neighborhood' in cleaned.columns:
    neighborhood_medians = cleaned.groupby('Neighborhood')['LotFrontage'].transform('median')
    cleaned['LotFrontage'] = cleaned['LotFrontage'].fillna(neighborhood_medians)
    cleaned['LotFrontage'] = cleaned['LotFrontage'].fillna(cleaned['LotFrontage'].median())

# A missing veneer area is zero when the dataset indicates no veneer.
if 'MasVnrArea' in cleaned.columns and 'MasVnrType' in cleaned.columns:
    no_veneer = cleaned['MasVnrType'].eq('None')
    cleaned.loc[no_veneer & cleaned['MasVnrArea'].isna(), 'MasVnrArea'] = 0

# Keep remaining missing values for model-pipeline imputation.
# In particular, GarageYrBlt and Electrical are not imputed here.

output_path = OUTPUT_DIR / 'ames_housing_clean.csv'
cleaned.to_csv(output_path, index=False)

print('Original shape:', df.shape)
print('Cleaned shape:', cleaned.shape)
print('Remaining missing cells:', int(cleaned.isna().sum().sum()))
print('\nRemaining missing values by column:')
display(cleaned.isna().sum()[lambda s: s > 0].sort_values(ascending=False))
print('\nSaved to:', output_path.resolve())